# LLM Tools and Use Cases:

Creating a simple Airline ticket booking agent

In [66]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display,Markdown
import gradio as gr
import json
import sqlite3

In [2]:
load_dotenv(override = True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("key loaded!")
else:
    print("key not loaded!")

key loaded!


In [3]:
openai = OpenAI()

In [ ]:
ticket_prices = {"delhi":"25000", "bangalore":"15000","chennai":"14000", "mumbai":"27000"}

def get_ticket_price(location):
    print(f"Tool called for city {location}")
    price = ticket_prices.get(location.lower(), "unknown ticket price")
    return f"The price of a ticket for the destination {location} is {price}"

print(get_ticket_price("Delhi"))

### Telling the model to utilise the tool: get_ticket_price:
* This is usually done by writing a particular dictionary "JSON" format for describing to the model, what tools to use.

In [5]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["location"],
        "additionalProperties": False
    }
}

In [6]:
tools = [{"type":"function","function":price_function}]

In [7]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'location': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['location'],
    'additionalProperties': False}}}]

In [8]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [53]:
def tool_chat(message, history):
    messages = [{"role":"system","content":system_message}]+history+[{"role":"user", "content":message}]
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages,
        tools= tools
    )

    while response.choices[0].finish_reason =="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        # extend => unpacks and appends an iterable to the list.
        messages.extend(responses)

        for message in messages:
            print(message)

        response = openai.chat.completions.create(model = "gpt-4.1-mini", messages =messages)

    return response.choices[0].message.content



In [54]:
# We have to write that function handle_tool_call:

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('location')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [55]:
gr.ChatInterface(fn = tool_chat, type = "messages").launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


Tool called for city Chennai
Tool called for city Bangalore
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'metadata': None, 'content': 'Hi', 'options': None}
{'role': 'assistant', 'metadata': None, 'content': 'Hello! How can I assist you with your flight today?', 'options': None}
{'role': 'user', 'content': 'Tell the flight ticket price difference between Chennai and Bangalor?'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_LeuPQ1haP9ZKefkP8Kve93H3', function=Function(arguments='{"location": "Chennai"}', name='get_ticket_price'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_I07XWrSjoowtQB5xMYBFxeQY', function=Function(arguments='{"location": "Bangalore"}', na

## Building Tool Calling for SQLite databse integration:

In [91]:
price_function_db = {
    "name": "get_ticket_price_db",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["location"],
        "additionalProperties": False
    }
}

In [92]:

tool_db = [{"type":"function","function":price_function_db}]

In [93]:
tool_db

[{'type': 'function',
  'function': {'name': 'get_ticket_price_db',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'location': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['location'],
    'additionalProperties': False}}}]

In [94]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)")
    conn.commit()


In [95]:
def get_ticket_price_db(city):
    print(f"Searching the database for finding the price of ticket for the {city}", flush = True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city!!"

In [96]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [97]:
ticket_prices = {"delhi":25000, "bangalore":15000,"chennai":14000, "mumbai":27000}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [98]:
def tool_chat2(message, history):
    messages = [{"role":"system","content":system_message}]+history+[{"role":"user", "content":message}]
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages,
        tools= tool_db
    )

    while response.choices[0].finish_reason =="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls2(message)
        messages.append(message)
        # extend => unpacks and appends an iterable to the list.
        messages.extend(responses)

        for message in messages:
            print(message)

        response = openai.chat.completions.create(model = "gpt-4.1-mini", messages =messages)

    return response.choices[0].message.content



In [99]:
# We have to write that function handle_tool_call:

def handle_tool_calls2(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price_db":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('location')
            price_details = get_ticket_price_db(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn = tool_chat2, type = "messages").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


Searching the database for finding the price of ticket for the Mumbai
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'metadata': None, 'content': 'Hi there ', 'options': None}
{'role': 'assistant', 'metadata': None, 'content': 'Hello! How can I assist you with your flight today?', 'options': None}
{'role': 'user', 'content': 'Give me the flight ticket price to Mumbai'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BIbRPoZ9SazQ5VHhpAXtdUgg', function=Function(arguments='{"location":"Mumbai"}', name='get_ticket_price_db'), type='function')])
{'role': 'tool', 'content': 'Ticket price to Mumbai is $27000.0', 'tool_call_id': 'call_BIbRPoZ9SazQ5VHhpAXtdUgg'}
